# Day 4 - Make It Safe to Run

The pipeline you built over the last three days runs. It produces output. Nobody raises an error.

That is not the same as it being safe to run.

A silent failure is when the pipeline produces wrong output without crashing. The report lands in someone's inbox and looks fine.

This session is about making those failures visible. You will add a set of checks to the HomeSphere pipeline - one at a time, with a clear decision about what each check protects against and what should happen if it fails.

---

## Pipeline setup

Run this cell first. It rebuilds the HomeSphere pipeline and gives you three DataFrames to work with:

- `silver_sales` - cleaned sales data
- `silver_products` - flattened product catalogue
- `gold_revenue` - joined table with line values calculated

In [ ]:
import pandas as pd
import json

# --- bronze: load raw sources ---
df_raw = pd.read_csv('../HomeSphere/data/sales_raw.csv')

with open('../HomeSphere/data/products_raw.json') as f:
    products_data = json.load(f)

# --- silver: clean sales ---
df = df_raw.copy()
df['unit_price'] = df['unit_price'].astype(str).str.replace('£', '', regex=False).astype(float)
df['order_date'] = pd.to_datetime(df['order_date'], format='mixed', dayfirst=True, errors='coerce')
df['quantity'] = pd.to_numeric(df['quantity'], errors='coerce')
df = df.dropna(subset=['quantity'])
df['quantity'] = df['quantity'].astype(int)
df['status'] = df['status'].str.lower().str.strip()
df = df.drop_duplicates()
df = df.dropna(subset=['product_id'])
df['region'] = df['region'].fillna('Unknown')
df = df[(df['unit_price'] > 0) & (df['quantity'] > 0)]
silver_sales = df.reset_index(drop=True)

# --- silver: flatten products ---
products = pd.json_normalize(products_data['products'])
products = products.rename(columns={
    'specs.rrp': 'rrp',
    'specs.warranty_years': 'warranty_years',
    'specs.colour': 'colour',
    'specs.connectivity': 'connectivity'
})
silver_products = products[['product_id', 'name', 'category']].copy()

# --- gold: join and calculate ---
gold_revenue = silver_sales.merge(silver_products, on='product_id', how='left')
gold_revenue['line_value'] = gold_revenue['quantity'] * gold_revenue['unit_price']

print(f'silver_sales:    {len(silver_sales)} rows')
print(f'silver_products: {len(silver_products)} rows')
print(f'gold_revenue:    {len(gold_revenue)} rows')

---

## The check pattern

Each check in this notebook has three parts:

- **Risk** - what could go wrong
- **Check** - what you test for
- **If this fails** - what the pipeline should do

The response is the most important decision. A check that prints a warning and carries on is not a check - it is a log message.

Use explicit logic that is readable and intentional:

```python
row_count = len(silver_sales)
if row_count < 20:
    raise ValueError(f"Row count too low: {row_count} rows. Pipeline stopped.")
```

Or collect failures and stop together:

```python
failures = []

if len(silver_sales) < 20:
    failures.append(f"Row count too low: {len(silver_sales)}")

if silver_sales["product_id"].isnull().sum() > 0:
    failures.append("Nulls found in product_id")

if failures:
    for msg in failures:
        print(f"FAIL: {msg}")
    raise ValueError("Validation failed - pipeline stopped.")
```

---

## TODO 1 - Row count

**Risk:** The sales file arrives with far fewer rows than expected - a truncated export, a failed transfer, or a filter applied upstream. The pipeline runs on the short file and produces understated revenue figures.

**Check:** `silver_sales` has at least 20 rows.

**If this fails:** Stop the pipeline. Do not promote to gold.

In [ ]:
row_count = len(silver_sales)

# TODO: check row_count and raise a clear error if it is too low


---

## TODO 2 - Null check on a key field

**Risk:** Some sales rows have no `product_id`. The left join will not crash - it will just produce nulls in the product columns for those rows. Revenue will be attributed to no category. Nobody will notice unless they look closely.

**Check:** No nulls in `product_id` in `silver_sales`.

**If this fails:** Stop the pipeline. Unmatched rows will silently understate category revenue.

In [ ]:
null_count = silver_sales["product_id"].isnull().sum()

# TODO: check null_count and raise a clear error if any are found


---

## TODO 3 - Join quality

**Risk:** A `product_id` in `silver_sales` has no match in `silver_products`. The left join keeps the sales row but leaves `category` as null. That order line is silently excluded from the revenue-by-category summary.

**Check:** No nulls in `category` after the join.

**If this fails:** Warn and show which order IDs were affected. Do not stop the pipeline silently - make the gap visible.

In [ ]:
unmatched = gold_revenue[gold_revenue["category"].isnull()]

# TODO: check whether unmatched is empty and respond clearly if not
# hint: show how many rows and which order_ids are affected


---

## TODO 4 - Value sanity

**Risk:** A negative or zero `quantity` or `line_value` would distort the revenue figures. This should have been caught in cleaning, but a check here protects against logic errors or changes upstream.

**Check:** All values in `quantity` and `line_value` are greater than zero.

**If this fails:** Collect the offending rows and stop promotion to gold.

In [ ]:
bad_rows = gold_revenue[
    (gold_revenue["quantity"] <= 0) | (gold_revenue["line_value"] <= 0)
]

# TODO: check whether bad_rows is empty and respond clearly if not
# hint: show how many rows failed and print them


---

## Stretch task - reusable functions

The four checks above each follow the same pattern. Move them into functions so they can be called on any DataFrame in the pipeline.

Fill in the two functions below, then call them on `silver_sales` and `gold_revenue`.

In [ ]:
def check_not_empty(df, name):
    # TODO: raise ValueError if df has no rows
    pass


def check_no_nulls(df, column, name):
    # TODO: raise ValueError if column has any nulls
    pass


# Call your functions:
check_not_empty(silver_sales, "silver_sales")
check_no_nulls(silver_sales, "product_id", "silver_sales")
check_not_empty(gold_revenue, "gold_revenue")
check_no_nulls(gold_revenue, "category", "gold_revenue")

print("All checks passed")

---

## Debrief

Look back at the four checks you wrote. For each one, the pipeline currently passes - the data is clean.

But the check is there for when it is not.

The question is: **what should happen when a check fails?**

Three options:

- **Stop** - raise an error and halt. Nothing downstream runs.
- **Warn and continue** - print the failure and keep going. Risky if downstream code uses the bad data.
- **Quarantine** - write the failing rows to a rejected file, continue with the clean rows.

For each of your four checks - which response makes most sense, and why? Does the answer change depending on who is consuming the output?